In [ ]:
import time
import itertools
import copy
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from genkit import GaussianFlowLinear
from genkit._noise import sample_scaled_isotropic_alpha_stable
from genkit.nn import MLPModel
from genkit.training import train
from genkit.metrics import mmd_rbf, sliced_wasserstein, tail_coverage_error


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, linear, rank=8, alpha=1.):
        super().__init__()
        self.linear = linear
        self.A = nn.Linear(linear.in_features, rank, bias=False)
        self.B = nn.Linear(rank, linear.out_features, bias=False)
        self.scale = alpha / rank

        for p in self.linear.parameters():
            p.requires_grad_(False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.linear(x) + self.scale * self.B(self.A(x))


class TailLoRAFlow(GaussianFlowLinear):
    def __init__(self, base, tau, rank=8, alpha=1., tail_weight=10., sharpness=5.):
        net = copy.deepcopy(base._net)
        for module in list(net.modules()):
            for name, child in module.named_children():
                if isinstance(child, nn.Linear):
                    setattr(module, name, LoRALinear(child, rank=rank, alpha=alpha))

        super().__init__(
            net, dim=base._dim, n_steps=base._n_steps, t_min=base._t_min, t_max=base._t_max,
            sigma_max=base._sigma_max, sampler=base._sampler, schedule=base._schedule,
            sample_steps=base._sample_steps, device=base._device, fdtype=base._fdtype, idtype=base._idtype,
        )
        self.tau = tau
        self.tail_weight = tail_weight
        self.sharpness = sharpness

    def _loss(self, x, z=None, t=None):
        v_hat, v, t = self._precompute_loss(x=x, z=z, t=t)
        s = x.flatten(1).norm(dim=1, keepdim=True)
        w = 1. + (self.tail_weight - 1.) * torch.sigmoid(self.sharpness * (s - self.tau))
        return w.view(-1, *([1] * (x.ndim - 1))) * F.mse_loss(v_hat, v, reduction="none")

In [ ]:
device = 'cpu'
dtype = torch.float32
batch_size = 512
lr = 5e-4
n_train = 25_000
n_epochs = 128
n_steps = 128
alpha = 1.7
dim = 3
train_data = sample_scaled_isotropic_alpha_stable(n_samples=n_train, dim=dim, alpha=alpha, device=device, dtype=dtype)
models, diagnostics = {}, {}

In [ ]:

start_time = time.time()
print(f"Train 'GFL'...", end="")

models["GFL"] = GaussianFlowLinear(MLPModel(dim=dim, width=128, depth=3), dim=dim, n_steps=n_steps, t_min=0., t_max=1., sampler="euler", sample_steps=n_steps, device=device, fdtype=dtype)
models["GFL"], diagnostics["GFL"] = train(models["GFL"], train_data, batch_size=batch_size, n_epochs=n_epochs, lr=lr, device=device, lr_schedule="constant")

print(f"done in {time.time() - start_time:.1f} s", flush=True)


In [ ]:
guided_kwargs = [("GFL+LoRA95",  dict(tau=torch.quantile(train_data.flatten(1).norm(dim=1), 0.95).item(),  rank=6, alpha=8., tail_weight=20., sharpness=5.)),
                 ("GFL+LoRA99",  dict(tau=torch.quantile(train_data.flatten(1).norm(dim=1), 0.99).item(),  rank=6, alpha=8., tail_weight=20., sharpness=5.)),
                 ]

for name, kwargs in guided_kwargs:
    start_time = time.time()
    print(f"Train '{name}'...", end="")

    models[name] = TailLoRAFlow(models["GFL"], **kwargs)
    models[name], diagnostics[name] = train(models[name], train_data, batch_size=batch_size, n_epochs=n_epochs, lr=lr, device=device, lr_schedule="constant")

    print(f"done in {time.time() - start_time:.1f} s", flush=True)

In [ ]:
n_tail = 100_000
n_pairwise = 1_000
n_trials = 2
sample_chunk_size = 10_000

tail_probs = torch.logspace(-1.0, -4.0, 8, device=device, dtype=dtype).tolist()
quantiles = [1.0 - p for p in tail_probs]
tce_cols = [f"tce_{q:.6f}" for q in quantiles]
probs = torch.tensor(tail_probs, device=device, dtype=dtype)

metric_rows = []

for trial in range(n_trials):
    print(f"Trial {trial + 1}/{n_trials}", flush=True)
    x_ref = sample_scaled_isotropic_alpha_stable(n_samples=n_tail, dim=dim, alpha=alpha, device=device, dtype=dtype)
    x_source = next(iter(models.values()))._sample_source(n_tail)

    for name, model in models.items():
        start_time = time.time()
        print(f"  Evaluating '{name}'...", end="")

        x_gen = model.sample(sample_source=x_source, chunk_size=sample_chunk_size)

        row = {"trial": trial + 1, "model": name}
        row["mmd_rbf"] = mmd_rbf(x_ref[:n_pairwise], x_gen[:n_pairwise])
        row["sliced_w2"] = sliced_wasserstein(x_ref[:n_pairwise], x_gen[:n_pairwise])

        tce = tail_coverage_error(x_ref, x_gen, probs=probs, tail="upper", mode="log", reduction="none")
        for col, err in zip(tce_cols, tce.mean(dim=1).cpu().numpy()):
            row[col] = float(err)

        metric_rows.append(row)
        print(f"done in {time.time() - start_time:.1f} s", flush=True)

df_trials = pd.DataFrame(metric_rows)

In [ ]:
df = df_trials.drop(columns="trial").groupby("model", sort=False).mean(numeric_only=True).reset_index()
df_std = df_trials.drop(columns="trial").groupby("model", sort=False).std(numeric_only=True).reset_index()
df_std_index = df_std.set_index("model")
model_width = int(df["model"].str.len().max())
markers = itertools.cycle(["o", "s", "^", "D", "v", "P", "X", "*"])

y_bounds = []
fig, ax = plt.subplots(figsize=(4, 4.5))

for marker, (_, r) in zip(markers, df.iterrows()):
    model = f"{r['model']:<{model_width}}"
    label = f"{model} | MMD={r['mmd_rbf']:.3g}, SW2={r['sliced_w2']:.3g}"

    y = r[tce_cols].to_numpy(dtype=float)
    std = df_std_index.loc[r["model"], tce_cols].fillna(0.0).to_numpy(dtype=float)
    lower = np.clip(y - std, 1e-12, None)
    upper = y + std

    line, = ax.plot(tail_probs, y, marker=marker, ms=6, lw=2.5, label=label, alpha=0.75)
    ax.fill_between(tail_probs, lower, upper, color=line.get_color(), alpha=0.14, linewidth=0, zorder=line.get_zorder() - 1)

    y_bounds.extend(y[np.isfinite(y) & (y > 0)].tolist())
    y_bounds.extend(upper[np.isfinite(upper) & (upper > 0)].tolist())

ax.set_xscale("log")
ax.invert_xaxis()
ax.set_yscale("log")
if y_bounds:
    ax.set_ylim(0.9 * min(y_bounds), 1.1 * max(y_bounds))
ax.set_xticks(tail_probs)
ax.set_xticklabels([f"{100 * q:.3f}%" for q in quantiles], rotation=45, ha="right")
ax.set_xlabel("Quantiles", fontsize=12)
ax.set_ylabel("TCE", fontsize=12)
ax.grid(alpha=0.3, which="both")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=1, frameon=False, prop={"family": "monospace", "size": 10})

plt.tight_layout(rect=(0, 0, 1, 0.86))
plt.show()